# Module 6 Assignment: Give Your Persona Agent More Tools

In Module 5 you built a Persona Agent with one tool. This assignment makes the same jump Lab 5
made on top of Lab 4: your agent goes from one tool to three, and `chat()` has to learn to route
between them instead of assuming.

You'll reuse your own Module 5 code at almost every step, plus the routing pattern from Lab 5.
This assignment is about **extending** what you already built, not writing it from scratch.

**What you'll build:**
1. Your persona, notes, and first tool, carried over from Module 5
2. A second tool, backed by a free, no-key external API
3. A third tool of your own design
4. JSON descriptions for all three tools
5. An updated system prompt that tells the model when to use each tool
6. A `call_tool(name, arguments)` dispatcher, exactly like Lab 5's
7. A full agent loop that routes across all three tools

**Provider:** Azure OpenAI via APIM, deployment `gpt-5.4-ptu`, same as every lab and assignment so
far.

If you can find where Lab 5 made this same change to Lab 4, you can make it here too.

## 0. Install the packages (run once)

Same as Module 5, plus one new package: `requests`, for calling your new tool's API.

In [2]:
!uv pip install --upgrade openai pypdf gradio python-dotenv requests

Resolved 63 packages in 314ms
Checked 63 packages in 5ms


## 1. Imports

**TODO:** Copy the imports from your Module 5 assignment, plus `requests` for calling an external
API. Add any other standard library module your second or third tool needs (for example `html`,
if the API you pick returns HTML-escaped text the way Open Trivia Database does).

In [48]:
# TODO: import load_dotenv, os, sys, json, PdfReader (from pypdf), gr (gradio), AzureOpenAI,
# and requests

from dotenv import load_dotenv
import os
import sys
import json
import requests
import html


from openai import AzureOpenAI
from pypdf import PdfReader
from IPython.display import Markdown, display
import gradio as gr


## 2. Connect to Azure OpenAI via APIM

Identical to every previous lab and assignment. Copy this cell directly from Module 5.

In [49]:
load_dotenv(override=True)

azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_api_key = os.getenv("AZURE_OPENAI_API_KEY")

if not all([azure_endpoint, azure_api_key]):
    sys.exit("Missing settings. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_API_KEY in your .env file.")

print(f"Azure OpenAI key exists and begins {azure_api_key[:8]}")

# The deployment we're using for this whole assignment - fixed, do not change it
DEPLOYMENT = "gpt-5.4-ptu"

azure_client = AzureOpenAI(
    azure_endpoint=azure_endpoint,
    api_key=azure_api_key,
    api_version="2024-10-21",
)

Azure OpenAI key exists and begins f4591325


## 3. Recreate your persona, notes, and first tool from Module 5

Bring forward everything you already built: your `persona`, your `notes/notes.pdf` loading code,
your one tool function from Module 5, and that tool's JSON description. Nothing here needs to
change, this is your starting point.

**TODO:** Paste in, from your Module 5 assignment:
- Your `persona` variable
- Your `PdfReader("notes/notes.pdf")` loading code that builds `notes`
- Your Module 5 tool function (rename it if you like, but keep track of its name)
- That tool's JSON schema dictionary

**Hint:** copy your `notes/notes.pdf` file into this assignment's folder too, in a `notes`
subfolder, exactly like Lab 5 did with Lab 4's notes file.

In [50]:
persona = (
    "TODO: You are BrainuLon, a hardcore AI study coach. Brainulon keeps quizzes short and"
    "fun,genuinely celebrates every right answer, and never makes a student feel bad about"
    "getting something wrong - just curious to help them learn it."
)

reader = PdfReader("notes/notes.pdf")
notes = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        notes += text

# TODO: paste your Module 5 tool function here, for example:
# def my_tool(value):
#     ...

def log_missed_question_tool(question):
    print(f"Tool called to log a missed question: {question}")
    with open("missed_questions.txt", "a", encoding="utf-8") as f:
        f.write(question + "\n")
    return "Missed question logged"

print(persona)
print(notes[:300], "...")

TODO: You are BrainuLon, a hardcore AI study coach. Brainulon keeps quizzes short andfun,genuinely celebrates every right answer, and never makes a student feel bad aboutgetting something wrong - just curious to help them learn it.
Bloodborne All Bosses Checklist
slaughter all of them
Regular Bosses
Cleric Beast
Father Gascoigne
Blood Starved Beast
Vicar Amelia
Witch of Hemwitch
Shadow of Yharman
Rom Vacuos Spider
Martyr Logarius
Darkbeast Paarl
The One Unborn
Celestial Emissary
Ebrietas
Amygdala
Micolash
Made with ChecklistMa ...


In [51]:
# TODO: paste your Module 5 tool's JSON schema dictionary here, for example:
# my_tool_json = {
#     "name": "my_tool",
#     "description": "...",
#     "parameters": { ... }
# }

log_missed_question_tool_json = {
    "name": "log_missed_question_tool",
    "description": "Use this tool to record a quiz question the student got wrong or didn't know, so they can review it later",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {"type": "string", "description": "The exact quiz question the student missed"}
        },
        "required": ["question"],
        "additionalProperties": False,
    },
}

tools = [{"type": "function", "function": log_missed_question_tool_json}]


## 4. New tool: something backed by a free API

Design a **second** tool for your Persona Agent, this one calling a free, no-key external API,
the same way Lab 5's `fetch_bonus_question_tool` calls the Open Trivia Database.

**Ideas, pick whichever fits your topic, or find your own:**
- A trivia-friendly agent could call the Open Trivia Database for a bonus question
- A weather-aware topic could call [Open-Meteo](https://open-meteo.com), a free weather API that
  needs no key
- A language-learning agent could call the [Free Dictionary API](https://dictionaryapi.dev) to
  define a word, though it is sometimes slow or briefly unavailable, so handle that the same way
  the next tip describes
- Any other free, no-key API relevant to your topic works too, just check it does not require
  signup or a paid key

**TODO:** Write your tool function below. It should make one `requests.get(...)` call inside a
`try`/`except` that catches `requests.exceptions.RequestException`, so a slow or unavailable API
returns a short, friendly string instead of crashing your notebook, exactly like Lab 5's tools
do.

In [52]:
def track_score_tool(correct):
    score = 0
    if os.path.exists("score.txt"):
        score = int(open("score.txt").read())

    if correct:
        score += 1

    open("score.txt", "w").write(str(score))

    return f"Score: {score}"
    pass

In [53]:
print(track_score_tool(True))   # Score: 1
print(track_score_tool(False))  # Score: 1 (no change)
print(track_score_tool(True))   # Score: 2

Score: 3
Score: 3
Score: 4


## 5. New tool: one more tool of your own design

Design a **third** tool. It can be another local tool in the spirit of Module 5 (a running score
tracker, a bookmarking tool for topics that need review, or an idea of your own), or a second
external API call if you would rather. Either is fine, as long as it does something the first two
tools do not already cover.

**TODO:** Write your third tool function below.

In [54]:


def bonus_question_tool():
    url = "https://opentdb.com/api.php?amount=1&category=15"
    data = requests.get(url).json()["results"][0]

    question = html.unescape(data["question"])
    answer = html.unescape(data["correct_answer"])

    return f"Question: {question}\nAnswer: {answer}"

pass

In [55]:
print(bonus_question_tool())  # quick manual test

Question: In the game "Fire Emblem: Shadow Dragon", what is the central protagonist's name?
Answer: Marth


## 6. Describe all three tools to the model with JSON

Same idea as Module 5 and Lab 5, just three schemas now. Remember an argument can be optional (an
empty `"required"` list), exactly like Lab 5's `fetch_bonus_question_tool`, if that fits your
tool.

In [56]:
# TODO: define api_tool_json and third_tool_json, the same shape as your Module 5 tool's
# JSON schema, then build the combined tools list:

# api_tool_json = { "name": "api_tool", "description": "...", "parameters": { ... } }
# third_tool_json = { "name": "third_tool", "description": "...", "parameters": { ... } }

# tools = [
#     {"type": "function", "function": my_tool_json},
#     {"type": "function", "function": api_tool_json},
#     {"type": "function", "function": third_tool_json},
# ]

track_score_tool_json = {
    "name": "track_score_tool",
    "description": "Use this tool every time the student answers a quiz question. Adds one point to their running score if the answer was correct. Wrong answers don't add points",
    "parameters": {
        "type": "object",
        "properties": {
            "correct": {"type": "boolean", "description": "True if the student's answer was correct, false if it was wrong"}
        },
        "required": ["correct"],
        "additionalProperties": False,
    },
}

bonus_question_tool_json = {
    "name": "bonus_question_tool",
    "description": "Use this tool when the student wants a bonus question or a break from the notes. Returns a random video game trivia question and its answer. Ask the question, and don't reveal the answer until the student has guessed",
    "parameters": {
        "type": "object",
        "properties": {},
        "required": [],
        "additionalProperties": False,
    },
}
log_missed_question_tool_json = {
    "name": "log_missed_question_tool",
    "description": "Use this tool to record a quiz question the student got wrong or didn't know, so they can review it later",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {"type": "string", "description": "The exact quiz question the student missed"}
        },
        "required": ["question"],
        "additionalProperties": False,
    },
}


tools = [
    {"type": "function", "function": track_score_tool_json},
    {"type": "function", "function": bonus_question_tool_json},
    {"type": "function", "function": log_missed_question_tool_json}
]


## 7. Update your system prompt for three tools

**TODO:** Build your `system_prompt` f-string the same way you did in Module 5, still using
`persona` and `notes`, but now with three rules telling the model when to reach for each tool,
the same way Lab 5's system prompt added two new sentences to Lab 4's.

In [57]:
system_prompt = f"""
{persona}

Here are the student's notes. Base all quiz questions on them:

{notes}

Rules for using your tools:
Use log_missed_question_tool to log any question missed, and any question the student does not know the answer to.
Use track_score_tool every time the student answers a quiz question, with correct set to true if they were right and false if they were wrong.
Use bonus_question_tool when the student asks for a bonus question or a break from the notes, then ask the question and don't reveal the answer until they have guessed.
Use bonus_question_tool for a random question to break monotony for the student after an unbroken chain of 5 questions from the notes.
"""

display(Markdown(system_prompt))


TODO: You are BrainuLon, a hardcore AI study coach. Brainulon keeps quizzes short andfun,genuinely celebrates every right answer, and never makes a student feel bad aboutgetting something wrong - just curious to help them learn it.

Here are the student's notes. Base all quiz questions on them:

Bloodborne All Bosses Checklist
slaughter all of them
Regular Bosses
Cleric Beast
Father Gascoigne
Blood Starved Beast
Vicar Amelia
Witch of Hemwitch
Shadow of Yharman
Rom Vacuos Spider
Martyr Logarius
Darkbeast Paarl
The One Unborn
Celestial Emissary
Ebrietas
Amygdala
Micolash
Made with ChecklistMaker.meMergos Wet Nurse
Gerhman First Hunter
Moon Presence
DLC Bosses
Ludwig, the Holy Blade
Orphan of Kos
Lady Maria of the Astral Clocktower
Living Failures
Laurence, the First Vicar
Chalice Dungeon Bosses
Undead Giant
Merciless Watcher
Watchdog of the Old Lords
Beast-Possessed Soul
Keeper of the Old Lords
Pthumerian Descendant
Maneater Boar
Made with ChecklistMaker.meBloodletting Beast
Brainsucker
Forgotten Madman
Pthumerian Elder
Abhorrent Beast
Loran Silverbeast
Loran Darkbeast
Yharnam, Pthumerian Queen
Blood-Starved Beast
Headed Bloodletting Beast
Amygdala
Celestial Emissary
Ebrieras, Daughter of the Cosmos
Rom, the Vacuous Spider
Made with ChecklistMaker.me

Rules for using your tools:
Use log_missed_question_tool to log any question missed, and any question the student does not know the answer to.
Use track_score_tool every time the student answers a quiz question, with correct set to true if they were right and false if they were wrong.
Use bonus_question_tool when the student asks for a bonus question or a break from the notes, then ask the question and don't reveal the answer until they have guessed.
Use bonus_question_tool for a random question to break monotony for the student after an unbroken chain of 5 questions from the notes.


## 8. The routing problem: which tool did the model actually call?

With three tools, `chat()` can no longer assume. It has to check `tool_call.function.name` and
call the matching Python function.

**TODO:** Copy the `call_tool(name, arguments)` dispatcher pattern from Lab 5, Step 9, but route
across your own three tool names instead.

In [63]:
def call_tool(name, arguments):
    if name == "track_score_tool":
        return track_score_tool(arguments["correct"])
    elif name == "bonus_question_tool":
        return bonus_question_tool()
    elif name == "log_missed_question_tool":
        return log_missed_question_tool(arguments["question"])
    return f"Unknown tool: {name}"
    pass

## 9. Update `chat()` to route across all three tools

**TODO:** Copy the finished agent loop from your Module 5 assignment. The `while` loop and the
`for tool_call in message_obj.tool_calls` loop do not change at all, the only new line is a call
to `call_tool(...)` in place of your old single hardcoded tool call.

In [64]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages, tools=tools)

    while response.choices[0].finish_reason == "tool_calls":
        message_obj = response.choices[0].message
        messages.append(message_obj)
        for tool_call in message_obj.tool_calls:
            name = tool_call.function.name
            arguments = json.loads(tool_call.function.arguments)
            result = call_tool(name, arguments)
            messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": result})
        response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages, tools=tools)

    return response.choices[0].message.content

## 10. Try your multi-tool Persona Agent

Launch the chat interface and try to trigger all three tools in one session: something that
should hit your original Module 5 tool, something that should hit your new API-backed tool, and
something that should hit your third tool. Watch the terminal or output to see which tool fires
each time. Interrupt the cell when you're done.

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Tool called to log a missed question: In the DLC Bosses list, who comes right after Ludwig, the Holy Blade?
Tool called to log a missed question: Which Regular Boss comes right after Vicar Amelia?
Tool called to log a missed question: In your Regular Bosses list, which boss comes right after Rom, Vacuous Spider?
Tool called to log a missed question: Which Chalice Dungeon Boss comes right after Keeper of the Old Lords?


## Reflection questions

Answer these in a new Markdown cell below.

1. What are your three tools, and what does each one do? Why did you choose the second and third
   tools you designed?


2. Compare this assignment to Module 5. What was exactly the same as going from one tool to more
   than one? What, if anything, surprised you?


3. Walk through what `call_tool` actually does the first time your agent calls your new
   API-backed tool. What would break if you deleted the `elif` branch for that tool?


4. Open Lab 5's system prompt and compare it to yours. Both add rules for two new tools, using
   only plain English sentences and no code changes to the agent loop. Why does that work, in
   terms of what a system prompt actually is?



1. 
I have a Missed question logger, a score tracker, and an API-backed bonus trivia question asker. I wanted to improve the agent I made in the previous lab to be able to handle more tool requests and be overall more useful for the purpose of lending the program to a friend I have who speedruns the game Bloodborne. I feel as thoughthe tools I added will improve the experience using the agent I already made.

2. 
The overall structure of the actual JSON stayed the same. It was literally just adding a few more lines in several places so that the agent could route to the correct tool. The biggest difference I noticed was the fact that I had to describe in more detail how and when to use each tool so the agent could better differentiate between them.

3. 
I'm unsure if I set it up properly because if I change the 'elif' to 'if' statements, the code still works the same. When you do call the tool though it's set to call all tools but not at once, it sends for the tool matching the name and argument requested.

4. 
Because it gives a message that the AI agent will refer back to and base all decisions off of. It's written so the model follows what is written for the prompt as guidelines for how to complete the task it has set out to accomplish. The agent could theoretically work wihout the prompt, but anything necessitating a hard guarantee, it must be coded and enforced.

## Wrap-up

Before submitting:

- Make sure your `.env` file is **not** included in what you submit, it contains your API key.
- Run the notebook top to bottom one more time to confirm it works cleanly.
- Answer all four reflection questions.
- Rename this file to include your name, for example `Module6Assignment_JaneDoe.ipynb`, and
  submit it.

You've now made the same jump Lab 5 made: one tool to several, with a real dispatcher and a loop
that already scaled to handle it.